# TFEX `USDM26` — live bid/ask from the Settrade Open API

This notebook connects to the **Settrade Open API (sandbox)** and pulls the **bid/ask order book**
for `USDM26` — the TFEX USD futures contract (1 contract = USD 1,000, cash-settled, last trade
29 Jun 2026). It is the *live-data* companion to Step 1, which used saved 1-minute candles.

> **Sandbox + read-only.** This uses the sandbox environment and only *reads* market data — it
> never places an order. Educational only, not investment advice.

**What you get:**
1. A one-shot **quote snapshot** (`get_quote_symbol`) — best bid, best ask, last, high/low.
2. The full **5-level bid/ask ladder** via the realtime feed (`subscribe_bid_offer`) — the same
   ladder you see on the trading screen.


## 1. Install (run once)

In [ ]:
# Already in this repo's uv environment. Uncomment if running elsewhere:
# !pip install settrade-v2 python-dotenv pandas

## 2. Imports

In [ ]:
import os
import json

import pandas as pd
from dotenv import load_dotenv
from settrade_v2 import Investor

## 3. Connect to the sandbox

Credentials come from `.env` (copy `.env.example` → `.env` and fill in your sandbox keys).

> **Gotcha worth knowing:** for the **sandbox**, the SDK expects `broker_id="098"` and
> `app_code="SANDBOX"`. If you pass `broker_id="SANDBOX"` you'll get `U-102: Invalid Signature`.
> The cell below forces the correct sandbox values, so it works even if `.env` has the wrong
> `SETTRADE_BROKER_ID`.

In [ ]:
load_dotenv(override=True)

APP_ID     = os.environ["SETTRADE_APP_ID"]
APP_SECRET = os.environ["SETTRADE_APP_SECRET"]

# Sandbox uses fixed values regardless of what's in .env:
IS_SANDBOX = os.environ.get("SETTRADE_ENV", "sandbox").lower() in ("sandbox", "uat")
APP_CODE  = "SANDBOX" if IS_SANDBOX else os.environ["SETTRADE_APP_CODE"]
BROKER_ID = "098"     if IS_SANDBOX else os.environ["SETTRADE_BROKER_ID"]

investor = Investor(
    app_id=APP_ID,
    app_secret=APP_SECRET,
    app_code=APP_CODE,
    broker_id=BROKER_ID,
    is_auto_queue=False,
)
mkt = investor.MarketData()
print(f"Connected. sandbox={IS_SANDBOX}  app_code={APP_CODE}  broker_id={BROKER_ID}")

## 4. Quote snapshot

`get_quote_symbol` returns one dict with the latest trade, day range, and top-of-book.
We print the raw response first so you can see every field this contract exposes.

In [ ]:
SYMBOL = "USDM26"

quote = mkt.get_quote_symbol(symbol=SYMBOL)
print(json.dumps(quote, indent=2, default=str))

### Best bid / ask and the spread

The spread is what a market-neutral trader pays to cross the book. We read it defensively
because field names can differ slightly between contract types.

In [ ]:
def first(d, *keys, default=None):
    """Return the first present, non-None key from a dict."""
    for k in keys:
        if isinstance(d, dict) and d.get(k) is not None:
            return d[k]
    return default

bid  = first(quote, "bid", "bidPrice", "bidPrice1")
ask  = first(quote, "ask", "askPrice", "askPrice1", "offer", "offerPrice")
last = first(quote, "last", "lastPrice", "price")
high = first(quote, "high", "highPrice")
low  = first(quote, "low", "lowPrice")

spread = (ask - bid) if (bid is not None and ask is not None) else None
mid    = ((ask + bid) / 2) if (bid is not None and ask is not None) else None

print(f"{SYMBOL}")
print(f"  best bid : {bid}")
print(f"  best ask : {ask}")
print(f"  spread   : {spread}")
print(f"  mid      : {mid}")
print(f"  last     : {last}   (day high {high} / low {low})")

## 5. The full bid/ask ladder (realtime feed)

`get_quote_symbol` gives the top of book; the **realtime** connection gives the full depth —
the multi-level ladder you see on the trading screen. We subscribe, grab the first message,
then unsubscribe.

`BidOfferV3` exposes `bid_price1..10` / `bid_volume1..10` and `ask_price1..10` / `ask_volume1..10`.

In [ ]:
import threading

rt = investor.RealtimeDataConnection()
_msg = {}
_got = threading.Event()

def on_bidoffer(message):
    # message is a dict (BidOfferV3 -> to_dict)
    _msg.update(message)
    _got.set()

sub = rt.subscribe_bid_offer(symbol=SYMBOL, on_message=on_bidoffer)
sub.start()

got = _got.wait(timeout=15)   # wait up to 15s for one snapshot
sub.stop()
print("received a ladder snapshot" if got else "no message in 15s (market may be closed)")

In [ ]:
LEVELS = 5

def g(d, key):
    return d.get(key)

rows = []
for i in range(1, LEVELS + 1):
    rows.append({
        "level": i,
        "bid_volume": g(_msg, f"bid_volume{i}"),
        "bid_price":  g(_msg, f"bid_price{i}"),
        "ask_price":  g(_msg, f"ask_price{i}"),
        "ask_volume": g(_msg, f"ask_volume{i}"),
    })

ladder = pd.DataFrame(rows).set_index("level")
ladder

## 6. Notes

- **Market hours.** Outside TFEX trading hours the ladder may be empty and bid/ask `None` —
  the snapshot's `last`/`high`/`low` still reflect the previous session.
- **Sandbox prices** are simulated and will differ from the live screen.
- **Read-only.** This notebook never calls `place_order`. See `settrade_open_api_setting.ipynb`
  for the (sandbox) order-placement example.
- **Next step.** Logging this bid/ask to a database over time is exactly what
  `THE_DATA_LOGGER.md` describes — the source of the real bid/ask in Steps 2, 4 and 5.